<a href="https://colab.research.google.com/github/ArmanveerKaur/Tweet-topic-classification/blob/main/Tweet_Topic_Classification.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

Deep Learning and Applications Project

TOPIC: Tweet Topic Classification

Submitted by:



*   Lakshay Mittal  (102215048)
*   Ayush Jindal    (102215129)
*   Armanveer Kaur  (102215151)







In [ ]:
import os
import pandas as pd
from sklearn.model_selection import train_test_split

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [ ]:
BASE_DIR = "/content/drive/MyDrive/twitter_topic_project"
RAW_DIR = os.path.join(BASE_DIR, "raw")
PROCESSED_DIR = os.path.join(BASE_DIR, "processed")

os.makedirs(RAW_DIR, exist_ok=True)
os.makedirs(PROCESSED_DIR, exist_ok=True)

print("Base directory:", BASE_DIR)

Base directory: /content/drive/MyDrive/twitter_topic_project


In [ ]:
import kagglehub
path = kagglehub.dataset_download("amananandrai/ag-news-classification-dataset")
print("Downloaded dataset to:", path)

train_path = os.path.join(path, "train.csv")
test_path  = os.path.join(path, "test.csv")

print("Train path:", train_path)
print("Test path:", test_path)

!cp "{train_path}" "{RAW_DIR}/train.csv"
!cp "{test_path}" "{RAW_DIR}/test.csv"

print("Raw files saved in:", RAW_DIR)
print("Raw dir contents:", os.listdir(RAW_DIR))


Using Colab cache for faster access to the 'ag-news-classification-dataset' dataset.
Downloaded dataset to: /kaggle/input/ag-news-classification-dataset
Train path: /kaggle/input/ag-news-classification-dataset/train.csv
Test path: /kaggle/input/ag-news-classification-dataset/test.csv
Raw files saved in: /content/drive/MyDrive/twitter_topic_project/raw
Raw dir contents: ['train.csv', 'test.csv']


In [ ]:
raw_train_df = pd.read_csv(os.path.join(RAW_DIR, "train.csv"))
raw_test_df  = pd.read_csv(os.path.join(RAW_DIR, "test.csv"))

print("Original train columns:", raw_train_df.columns.tolist())
print("Original test  columns:", raw_test_df.columns.tolist())

def normalize_columns(df):
    cols = list(df.columns)
    if "Class Index" in cols:  #
        df = df.rename(columns={
            "Class Index": "label",
            "Title": "title",
            "Description": "description"
        })
    else:
        df = df.rename(columns={
            cols[0]: "label",
            cols[1]: "title",
            cols[2]: "description"
        })
    return df

train_df = normalize_columns(raw_train_df)
test_df  = normalize_columns(raw_test_df)

print("Normalized train columns:", train_df.columns.tolist())
print("Normalized test  columns:", test_df.columns.tolist())

print("Train shape:", train_df.shape)
print("Test shape:", test_df.shape)

for df in (train_df, test_df):
    df["title"] = df["title"].astype(str)
    df["description"] = df["description"].astype(str)
    df["text"] = df["title"] + " " + df["description"]

print("\nSample combined text:")
print(train_df[["label", "text"]].head())

Original train columns: ['Class Index', 'Title', 'Description']
Original test  columns: ['Class Index', 'Title', 'Description']
Normalized train columns: ['label', 'title', 'description']
Normalized test  columns: ['label', 'title', 'description']
Train shape: (120000, 3)
Test shape: (7600, 3)

Sample combined text:
   label                                               text
0      3  Wall St. Bears Claw Back Into the Black (Reute...
1      3  Carlyle Looks Toward Commercial Aerospace (Reu...
2      3  Oil and Economy Cloud Stocks' Outlook (Reuters...
3      3  Iraq Halts Oil Exports from Main Southern Pipe...
4      3  Oil prices soar to all-time record, posing new...


In [ ]:
for df in (train_df, test_df):
    df["label"] = df["label"].astype(int)
    df["label_id"] = df["label"] - 1

print("\nLabel mapping check:")
print(train_df[["label", "label_id"]].head())

print("\nLabel distribution (normalized):")
print(train_df["label_id"].value_counts(normalize=True))


Label mapping check:
   label  label_id
0      3         2
1      3         2
2      3         2
3      3         2
4      3         2

Label distribution (normalized):
label_id
2    0.25
3    0.25
1    0.25
0    0.25
Name: proportion, dtype: float64


In [ ]:
X = train_df["text"].values
y = train_df["label_id"].values

X_train, X_val, y_train, y_val = train_test_split(
    X,
    y,
    test_size=0.1,
    random_state=42,
    stratify=y
)

X_test = test_df["text"].values
y_test = test_df["label_id"].values

print("\nSizes:")
print("Train:", len(X_train))
print("Val:", len(X_val))
print("Test:", len(X_test))


Sizes:
Train: 108000
Val: 12000
Test: 7600


In [ ]:
train_clean = pd.DataFrame({"text": X_train, "label_id": y_train})
val_clean   = pd.DataFrame({"text": X_val,   "label_id": y_val})
test_clean  = pd.DataFrame({"text": X_test,  "label_id": y_test})

train_clean.to_csv(os.path.join(PROCESSED_DIR, "train_clean.csv"), index=False)
val_clean.to_csv(os.path.join(PROCESSED_DIR, "val_clean.csv"), index=False)
test_clean.to_csv(os.path.join(PROCESSED_DIR, "test_clean.csv"), index=False)

print("\nProcessed files saved in:", PROCESSED_DIR)
print(os.listdir(PROCESSED_DIR))


Processed files saved in: /content/drive/MyDrive/twitter_topic_project/processed
['train_clean.csv', 'val_clean.csv', 'test_clean.csv']


tf-idf+ logostic regression


In [ ]:
import os
import pandas as pd
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import f1_score, classification_report, confusion_matrix

from google.colab import drive
drive.mount('/content/drive')

BASE_DIR = "/content/drive/MyDrive/twitter_topic_project"
PROCESSED_DIR = os.path.join(BASE_DIR, "processed")

train_clean_path = os.path.join(PROCESSED_DIR, "train_clean.csv")
val_clean_path   = os.path.join(PROCESSED_DIR, "val_clean.csv")
test_clean_path  = os.path.join(PROCESSED_DIR, "test_clean.csv")

train_df = pd.read_csv(train_clean_path)
val_df   = pd.read_csv(val_clean_path)
test_df  = pd.read_csv(test_clean_path)

print("Train:", train_df.shape, " Val:", val_df.shape, " Test:", test_df.shape)

X_train, y_train = train_df["text"].values, train_df["label_id"].values
X_val,   y_val   = val_df["text"].values,   val_df["label_id"].values
X_test,  y_test  = test_df["text"].values,  test_df["label_id"].values

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
Train: (108000, 2)  Val: (12000, 2)  Test: (7600, 2)


In [ ]:
tfidf = TfidfVectorizer(
    max_features=20000,
    ngram_range=(1, 2),
    stop_words="english"
)

print("TF-IDF")
X_tfidf_train = tfidf.fit_transform(list(X_train) + list(X_val))
y_tfidf_train = list(y_train) + list(y_val)

X_tfidf_test = tfidf.transform(X_test)

print("TF-IDF train shape:", X_tfidf_train.shape)
print("TF-IDF test shape:", X_tfidf_test.shape)

TF-IDF
TF-IDF train shape: (120000, 20000)
TF-IDF test shape: (7600, 20000)


In [ ]:
logreg = LogisticRegression(
    max_iter=1000,
    n_jobs=-1,
    verbose=1
)

In [ ]:
print("Training Logistic Regression")
logreg.fit(X_tfidf_train, y_tfidf_train)

y_pred_test = logreg.predict(X_tfidf_test)

macro_f1 = f1_score(y_test, y_pred_test, average="macro")
print("\nTF-IDF + Logistic Regression")
print("Macro F1 on test set:", macro_f1)

print("\nClassification Report:")
print(classification_report(y_test, y_pred_test))

print("\nConfusion Matrix:")
print(confusion_matrix(y_test, y_pred_test))

Training Logistic Regression


[Parallel(n_jobs=-1)]: Using backend LokyBackend with 2 concurrent workers.



TF-IDF + Logistic Regression
Macro F1 on test set: 0.9196832313655389

Classification Report:
              precision    recall  f1-score   support

           0       0.93      0.91      0.92      1900
           1       0.95      0.98      0.97      1900
           2       0.89      0.89      0.89      1900
           3       0.90      0.90      0.90      1900

    accuracy                           0.92      7600
   macro avg       0.92      0.92      0.92      7600
weighted avg       0.92      0.92      0.92      7600


Confusion Matrix:
[[1732   53   67   48]
 [  17 1865   12    6]
 [  59   14 1684  143]
 [  49   21  120 1710]]


In [ ]:
RESULTS_DIR = os.path.join(BASE_DIR, "results")
os.makedirs(RESULTS_DIR, exist_ok=True)

results_df = pd.DataFrame({
    "text": X_test,
    "true_label": y_test,
    "pred_label": y_pred_test
})

results_df.to_csv(os.path.join(RESULTS_DIR, "baseline_tfidf_logreg_predictions.csv"), index=False)

with open(os.path.join(RESULTS_DIR, "baseline_tfidf_logreg_f1.txt"), "w") as f:
    f.write(f"Macro F1 (TF-IDF + Logistic Regression) on test set: {macro_f1:.4f}\n")

print("\nSaved baseline predictions and F1 to:", RESULTS_DIR)


Saved baseline predictions and F1 to: /content/drive/MyDrive/twitter_topic_project/results


BiLSTM

In [ ]:
PROCESSED_DIR = os.path.join(BASE_DIR, "processed")
RESULTS_DIR = os.path.join(BASE_DIR, "results_bilstm")
os.makedirs(RESULTS_DIR, exist_ok=True)

train_df = pd.read_csv(os.path.join(PROCESSED_DIR, "train_clean.csv"))
val_df   = pd.read_csv(os.path.join(PROCESSED_DIR, "val_clean.csv"))
test_df  = pd.read_csv(os.path.join(PROCESSED_DIR, "test_clean.csv"))

print("Train:", train_df.shape, " Val:", val_df.shape, " Test:", test_df.shape)

X_train, y_train = train_df["text"].astype(str).values, train_df["label_id"].astype(int).values
X_val,   y_val   = val_df["text"].astype(str).values,   val_df["label_id"].astype(int).values
X_test,  y_test  = test_df["text"].astype(str).values,  test_df["label_id"].astype(int).values

Train: (108000, 2)  Val: (12000, 2)  Test: (7600, 2)


In [ ]:
import os
import pandas as pd
import numpy as np

In [ ]:
PROCESSED_DIR = os.path.join(BASE_DIR, "processed")
RESULTS_DIR = os.path.join(BASE_DIR, "results_bilstm")
os.makedirs(RESULTS_DIR, exist_ok=True)

train_df = pd.read_csv(os.path.join(PROCESSED_DIR, "train_clean.csv"))
val_df   = pd.read_csv(os.path.join(PROCESSED_DIR, "val_clean.csv"))
test_df  = pd.read_csv(os.path.join(PROCESSED_DIR, "test_clean.csv"))

print("Train:", train_df.shape, " Val:", val_df.shape, " Test:", test_df.shape)

X_train, y_train = train_df["text"].astype(str).values, train_df["label_id"].astype(int).values
X_val,   y_val   = val_df["text"].astype(str).values,   val_df["label_id"].astype(int).values
X_test,  y_test  = test_df["text"].astype(str).values,  test_df["label_id"].astype(int).values

from tensorflow.keras.preprocessing.text import Tokenizer
from tensorflow.keras.preprocessing.sequence import pad_sequences

max_words = 20000
max_len   = 100

tokenizer = Tokenizer(num_words=max_words, oov_token="<OOV>")
tokenizer.fit_on_texts(list(X_train) + list(X_val))

def texts_to_padded(texts):
    seqs = tokenizer.texts_to_sequences(texts)
    return pad_sequences(seqs, maxlen=max_len, padding="post", truncating="post")

X_train_seq = texts_to_padded(X_train)
X_val_seq   = texts_to_padded(X_val)
X_test_seq  = texts_to_padded(X_test)

print("Train seq shape:", X_train_seq.shape)
print("Val seq shape:",   X_val_seq.shape)
print("Test seq shape:",  X_test_seq.shape)

vocab_size = min(max_words, len(tokenizer.word_index) + 1)
print("Vocab size used:", vocab_size)

Train: (108000, 2)  Val: (12000, 2)  Test: (7600, 2)
Train seq shape: (108000, 100)
Val seq shape: (12000, 100)
Test seq shape: (7600, 100)
Vocab size used: 20000


In [ ]:
import tensorflow as tf
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Embedding, Bidirectional, LSTM, Dense, Dropout

embedding_dim = 128
lstm_units = 128
num_classes = 4

model = Sequential([
    tf.keras.layers.Input(shape=(max_len,), dtype="int32"),
    Embedding(input_dim=vocab_size, output_dim=embedding_dim),
    Bidirectional(LSTM(lstm_units, return_sequences=False)),
    Dropout(0.3),
    Dense(64, activation="relu"),
    Dropout(0.3),
    Dense(num_classes, activation="softmax")
])

model.compile(
    loss="sparse_categorical_crossentropy",
    optimizer="adam",
    metrics=["accuracy"]
)

model.summary()


Model: "sequential"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ embedding (Embedding)           │ (None, 100, 128)       │     2,560,000 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ bidirectional (Bidirectional)   │ (None, 256)            │       263,168 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout (Dropout)               │ (None, 256)            │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense (Dense)                   │ (None, 64)             │        16,448 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout_1 (Dropout)             │ (None, 64)             │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_1 (Dense)                 │ (None, 4)              │           260 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 2,839,876 (10.83 MB)

 Trainable params: 2,839,876 (10.83 MB)

 Non-trainable params: 0 (0.00 B)

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

import os, pandas as pd

BASE_DIR = "/content/drive/MyDrive/twitter_topic_project"
PROCESSED_DIR = os.path.join(BASE_DIR, "processed")
print("BASE_DIR:", BASE_DIR)
print("PROCESSED_DIR contents:", os.listdir(PROCESSED_DIR))

train_df = pd.read_csv(os.path.join(PROCESSED_DIR, "train_clean.csv"))
val_df   = pd.read_csv(os.path.join(PROCESSED_DIR, "val_clean.csv"))
test_df  = pd.read_csv(os.path.join(PROCESSED_DIR, "test_clean.csv"))
print("Train/Val/Test shapes:", train_df.shape, val_df.shape, test_df.shape)


Mounted at /content/drive
BASE_DIR: /content/drive/MyDrive/twitter_topic_project
PROCESSED_DIR contents: ['train_clean.csv', 'val_clean.csv', 'test_clean.csv']
Train/Val/Test shapes: (108000, 2) (12000, 2) (7600, 2)


In [ ]:
from tensorflow.keras.callbacks import EarlyStopping, ModelCheckpoint

checkpoint_path = os.path.join(RESULTS_DIR, "bilstm_best.weights.h5")

early_stop = EarlyStopping(
    monitor="val_loss",
    patience=2,
    restore_best_weights=True
)

checkpoint = ModelCheckpoint(
    checkpoint_path,
    monitor="val_loss",
    save_best_only=True,
    save_weights_only=True,
    verbose=1
)

history = model.fit(
    X_train_seq, y_train,
    validation_data=(X_val_seq, y_val),
    epochs=10,
    batch_size=128,
    callbacks=[early_stop, checkpoint],
    verbose=1
)

print("Training done. Best weights saved to:", checkpoint_path)

Epoch 1/10
843/844 ━━━━━━━━━━━━━━━━━━━━ 0s 20ms/step - accuracy: 0.7714 - loss: 0.5839
Epoch 1: val_loss improved from inf to 0.24156, saving model to /content/drive/MyDrive/twitter_topic_project/results_bilstm/bilstm_best.weights.h5
844/844 ━━━━━━━━━━━━━━━━━━━━ 24s 21ms/step - accuracy: 0.7716 - loss: 0.5834 - val_accuracy: 0.9192 - val_loss: 0.2416
Epoch 2/10
843/844 ━━━━━━━━━━━━━━━━━━━━ 0s 19ms/step - accuracy: 0.9374 - loss: 0.1952
Epoch 2: val_loss did not improve from 0.24156
844/844 ━━━━━━━━━━━━━━━━━━━━ 17s 20ms/step - accuracy: 0.9373 - loss: 0.1952 - val_accuracy: 0.9216 - val_loss: 0.2482
Epoch 3/10
842/844 ━━━━━━━━━━━━━━━━━━━━ 0s 20ms/step - accuracy: 0.9486 - loss: 0.1478
Epoch 3: val_loss did not improve from 0.24156
844/844 ━━━━━━━━━━━━━━━━━━━━ 18s 21ms/step - accuracy: 0.9486 - loss: 0.1478 - val_accuracy: 0.9215 - val_loss: 0.2692
Training done. Best weights saved to: /content/drive/MyDrive/twitter_topic_project/results_bilstm/bilstm_best.weights.h5


In [ ]:
from sklearn.metrics import f1_score, classification_report, confusion_matrix

model.load_weights(checkpoint_path)

y_test_probs = model.predict(X_test_seq, batch_size=256)
y_test_pred  = np.argmax(y_test_probs, axis=1)

macro_f1_bilstm = f1_score(y_test, y_test_pred, average="macro")
print("\nBiLSTM BASELINE RESULTS")
print("BiLSTM Macro F1 on test set:", macro_f1_bilstm)

print("\nClassification Report (BiLSTM):")
print(classification_report(y_test, y_test_pred))

print("\nConfusion Matrix (BiLSTM):")
print(confusion_matrix(y_test, y_test_pred))

pred_df = pd.DataFrame({
    "text": X_test,
    "true_label": y_test,
    "pred_label": y_test_pred
})
pred_df.to_csv(os.path.join(RESULTS_DIR, "bilstm_predictions.csv"), index=False)

with open(os.path.join(RESULTS_DIR, "bilstm_macro_f1.txt"), "w") as f:
    f.write(f"BiLSTM Macro F1 on test set: {macro_f1_bilstm:.4f}\n")

print("\nSaved BiLSTM predictions + F1 to:", RESULTS_DIR)


30/30 ━━━━━━━━━━━━━━━━━━━━ 1s 32ms/step

BiLSTM BASELINE RESULTS
BiLSTM Macro F1 on test set: 0.9157625726619436

Classification Report (BiLSTM):
              precision    recall  f1-score   support

           0       0.96      0.88      0.92      1900
           1       0.96      0.97      0.97      1900
           2       0.85      0.91      0.88      1900
           3       0.90      0.89      0.89      1900

    accuracy                           0.92      7600
   macro avg       0.92      0.92      0.92      7600
weighted avg       0.92      0.92      0.92      7600


Confusion Matrix (BiLSTM):
[[1678   54  118   50]
 [   9 1851   27   13]
 [  26    9 1732  133]
 [  38   11  154 1697]]

Saved BiLSTM predictions + F1 to: /content/drive/MyDrive/twitter_topic_project/results_bilstm


In [ ]:
RESULTS_BERT = os.path.join(BASE_DIR, "results_bert")
os.makedirs(RESULTS_BERT, exist_ok=True)


BERT

In [ ]:
!pip install -q transformers datasets accelerate

from sklearn.metrics import f1_score, classification_report, confusion_matrix
import torch
from torch.utils.data import Dataset
from transformers import (
    AutoTokenizer,
    AutoModelForSequenceClassification,
    TrainingArguments,
    Trainer
)


In [ ]:
wtrain_df = pd.read_csv(os.path.join(PROCESSED_DIR, "train_clean.csv"))
val_df   = pd.read_csv(os.path.join(PROCESSED_DIR, "val_clean.csv"))
test_df  = pd.read_csv(os.path.join(PROCESSED_DIR, "test_clean.csv"))

print("Train:", train_df.shape, " Val:", val_df.shape, " Test:", test_df.shape)

for df in (train_df, val_df, test_df):
    df["label_id"] = df["label_id"].astype(int)
    df["text"] = df["text"].astype(str)

model_name = "bert-base-uncased"
num_labels = 4

tokenizer = AutoTokenizer.from_pretrained(model_name)

class NewsDataset(Dataset):
    def __init__(self, texts, labels, tokenizer, max_length=128):
        self.texts = list(texts)
        self.labels = list(labels)
        self.tokenizer = tokenizer
        self.max_length = max_length

    def __getitem__(self, idx):
        enc = self.tokenizer(
            self.texts[idx],
            padding="max_length",
            truncation=True,
            max_length=self.max_length,
            return_tensors="pt"
        )
        item = {k: v.squeeze(0) for k, v in enc.items()}
        item["labels"] = torch.tensor(self.labels[idx], dtype=torch.long)
        return item

    def __len__(self):
        return len(self.texts)

train_dataset = NewsDataset(train_df.text, train_df.label_id, tokenizer)
val_dataset   = NewsDataset(val_df.text,   val_df.label_id, tokenizer)
test_dataset  = NewsDataset(test_df.text,  test_df.label_id, tokenizer)

Train: (108000, 2)  Val: (12000, 2)  Test: (7600, 2)


/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


tokenizer_config.json:   0%|          | 0.00/48.0 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/570 [00:00<?, ?B/s]

vocab.txt:   0%|          | 0.00/232k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/466k [00:00<?, ?B/s]

In [ ]:
device = "cuda" if torch.cuda.is_available() else "cpu"
print("Device:", device)

model = AutoModelForSequenceClassification.from_pretrained(
    model_name,
    num_labels=num_labels
).to(device)


Device: cuda


model.safetensors:   0%|          | 0.00/440M [00:00<?, ?B/s]

Some weights of BertForSequenceClassification were not initialized from the model checkpoint at bert-base-uncased and are newly initialized: ['classifier.bias', 'classifier.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


In [ ]:
RESULTS_BERT = os.path.join(BASE_DIR, "results_bert")
os.makedirs(RESULTS_BERT, exist_ok=True)

def compute_metrics(eval_pred):
    logits, labels = eval_pred
    preds = np.argmax(logits, axis=-1)
    return {"macro_f1": f1_score(labels, preds, average="macro")}

training_args = TrainingArguments(
    output_dir=os.path.join(BASE_DIR, "models_bert"),
    eval_strategy="epoch",          # <-- FIXED
    save_strategy="epoch",
    load_best_model_at_end=True,
    per_device_train_batch_size=16,
    per_device_eval_batch_size=32,
    num_train_epochs=1,
    learning_rate=2e-5,
    metric_for_best_model="macro_f1",
    greater_is_better=True,
    logging_steps=200,
    logging_strategy="steps",
    report_to="none"
)

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=train_dataset,
    eval_dataset=val_dataset,
    tokenizer=tokenizer,
    compute_metrics=compute_metrics
)

/tmp/ipython-input-578961861.py:25: FutureWarning: `tokenizer` is deprecated and will be removed in version 5.0.0 for `Trainer.__init__`. Use `processing_class` instead.
  trainer = Trainer(


In [ ]:
import numpy as np

In [ ]:
trainer.train()

Epoch,Training Loss,Validation Loss,Macro F1
1,0.073600,0.224668,0.948136


TrainOutput(global_step=6750, training_loss=0.10977719455295139, metrics={'train_runtime': 2397.6875, 'train_samples_per_second': 45.043, 'train_steps_per_second': 2.815, 'total_flos': 7104126062592000.0, 'train_loss': 0.10977719455295139, 'epoch': 1.0})

In [ ]:
test_results = trainer.evaluate(test_dataset)
bert_f1 = test_results["eval_macro_f1"]
print("\nBERT TEST RESULTS")
print(test_results)
print("\nBERT Macro F1:", bert_f1)

pred_output = trainer.predict(test_dataset)
test_logits = pred_output.predictions
test_preds = np.argmax(test_logits, axis=-1)
test_labels = pred_output.label_ids

label_map = {0:"World", 1:"Sports", 2:"Business", 3:"Sci/Tech"}

true_topics_str = [label_map[int(y)] for y in test_labels]
pred_topics_str = [label_map[int(y)] for y in test_preds]

print("\nBERT PREDICTIONS")
for i in range(50):
    print(f"\nText: {test_df['text'].iloc[i]}")
    print(f"True Topic: {true_topics_str[i]}")
    print(f"Predicted Topic: {pred_topics_str[i]}")


BERT TEST RESULTS
{'eval_loss': 0.24403385818004608, 'eval_macro_f1': 0.9436104507162371, 'eval_runtime': 51.6189, 'eval_samples_per_second': 147.233, 'eval_steps_per_second': 4.611, 'epoch': 1.0}

BERT Macro F1: 0.9436104507162371

BERT PREDICTIONS

Text: Fears for T N pension after talks Unions representing workers at Turner   Newall say they are 'disappointed' after talks with stricken parent firm Federal Mogul.
True Topic: Business
Predicted Topic: Business

Text: The Race is On: Second Private Team Sets Launch Date for Human Spaceflight (SPACE.com) SPACE.com - TORONTO, Canada -- A second\team of rocketeers competing for the  #36;10 million Ansari X Prize, a contest for\privately funded suborbital space flight, has officially announced the first\launch date for its manned rocket.
True Topic: Sci/Tech
Predicted Topic: Sci/Tech

Text: Ky. Company Wins Grant to Study Peptides (AP) AP - A company founded by a chemistry researcher at the University of Louisville won a grant to develop 

In [ ]:
pred_df = pd.DataFrame({
    "text": test_df.text.values,
    "true_label_id": test_labels,
    "pred_label_id": test_preds
})

pred_df.to_csv(os.path.join(RESULTS_BERT, "bert_predictions.csv"), index=False)

with open(os.path.join(RESULTS_BERT, "bert_macro_f1.txt"), "w") as f:
    f.write(f"{bert_f1:.4f}")

print("\nSaved BERT outputs to:", RESULTS_BERT)



Saved BERT outputs to: /content/drive/MyDrive/twitter_topic_project/results_bert


In [ ]:
label_map = {0: "World", 1: "Sports", 2: "Business", 3: "Sci/Tech"}

In [ ]:
RESULTS_BERT = os.path.join(BASE_DIR, "results_bert2")
os.makedirs(RESULTS_BERT, exist_ok=True)

In [ ]:
true_label_str = [label_map[int(x)] for x in test_labels]
pred_label_str = [label_map[int(x)] for x in test_preds]

pred_df = pd.DataFrame({
    "text": test_df.text.values,
    "true_label": true_label_str,
    "pred_label": pred_label_str
})

pred_df.to_csv(os.path.join(RESULTS_BERT, "bert_fast_predictions_readable.csv"), index=False)

with open(os.path.join(RESULTS_BERT, "bert_fast_macro_f1.txt"), "w") as f:
    f.write(f"{bert_f1:.4f}\n")

print("\nSaved FAST BERT readable predictions to:", RESULTS_BERT)



Saved FAST BERT readable predictions to: /content/drive/MyDrive/twitter_topic_project/results_bert2
